# ForgeGraph AI - Dataset exploration

This is the first notebook for ForgeGraph. The goal is to understand the structure of the MaintIE dataset before trying any model training.

**Dataset:** [MaintIE](https://github.com/nlp-tlp/maintie) (Bikaun et al., 2024)  
**Paper:** [LREC-COLING 2024](https://aclanthology.org/2024.lrec-main.954/)  
**File:** `data/gold_release.json` from the authors' repository.

We'll inspect records, entity labels, and relationships. This notebook doesn't train a model.

## 1. Load the data

Each record is a short maintenance note with tokens, entity annotations, and relations. We can load it directly from the original public GitHub repository. No libraries need to be installed for this notebook.

In [1]:
import json
from collections import Counter
from urllib.request import urlopen

DATA_URL = "https://raw.githubusercontent.com/nlp-tlp/maintie/main/data/gold_release.json"

with urlopen(DATA_URL, timeout=30) as response:
    records = json.load(response)

print("Number of records:", len(records))
print("Fields in first record:", list(records[0].keys()))

Number of records: 1076
Fields in first record: ['text', 'tokens', 'entities', 'relations']


## 2. Look at an actual maintenance record

We'll print one record and its annotations. In MaintIE, entity `start` is included and `end` is excluded, so `tokens[start:end]` gives the annotated phrase.

In [2]:
record = records[0]

print("Text:", record["text"])
print("Tokens:", record["tokens"])

print("\nEntities:")
for entity in record["entities"]:
    phrase = " ".join(record["tokens"][entity["start"]:entity["end"]])
    print(f"  {phrase!r} -> {entity['type']}")

print("\nRelations:")
for relation in record["relations"]:
    source = record["entities"][relation["head"]]
    target = record["entities"][relation["tail"]]
    source_text = " ".join(record["tokens"][source["start"]:source["end"]])
    target_text = " ".join(record["tokens"][target["start"]:target["end"]])
    print(f"  {source_text!r} --{relation['type']}--> {target_text!r}")

Text: <id> air conditioner thermostat not working
Tokens: ['<id>', 'air', 'conditioner', 'thermostat', 'not', 'working']

Entities:
  'air conditioner' -> PhysicalObject/EmittingObject/ElectricCoolingObject
  'thermostat' -> PhysicalObject/SensingObject/TemperatureSensingObject
  'not working' -> State/UndesirableState/FailedState

Relations:
  'air conditioner' --hasPart--> 'thermostat'
  'not working' --hasParticipant/hasPatient--> 'thermostat'


## 3. Count entity categories

MaintIE has detailed labels such as `PhysicalObject/SensingObject/TemperatureSensingObject`. For a simple overview, we'll count only the first part of each label, like `PhysicalObject`.

In [3]:
entity_counts = Counter()
for record in records:
    for entity in record["entities"]:
        top_level_label = entity["type"].split("/")[0]
        entity_counts[top_level_label] += 1

print("Top-level entity counts:")
for label, count in entity_counts.most_common():
    print(f"  {label}: {count}")

Top-level entity counts:
  PhysicalObject: 1994
  Activity: 784
  State: 438
  Process: 146
  Property: 35


## 4. Count relation types

Relations describe connections between pairs of entities. Let's see which types occur most often.

In [4]:
relation_counts = Counter()
for record in records:
    for relation in record["relations"]:
        relation_counts[relation["type"]] += 1

print("Relation counts:")
for label, count in relation_counts.most_common():
    print(f"  {label}: {count}")

Relation counts:
  hasParticipant/hasPatient: 1206
  hasPart: 533
  isA: 364
  hasParticipant/hasAgent: 166
  contains: 38
  hasProperty: 34


## 5. Check the annotation indices

Before model training, it is useful to check whether entity spans and relation indices point to valid entries. This is a basic data-quality check, not a full validation of the annotations.

In [5]:
invalid_spans = 0
invalid_relations = 0

for record in records:
    tokens = record["tokens"]
    entities = record["entities"]

    for entity in entities:
        if not (0 <= entity["start"] < entity["end"] <= len(tokens)):
            invalid_spans += 1

    for relation in record["relations"]:
        if not (0 <= relation["head"] < len(entities)
                and 0 <= relation["tail"] < len(entities)):
            invalid_relations += 1

print("Invalid entity spans:", invalid_spans)
print("Invalid relation indices:", invalid_relations)

Invalid entity spans: 0
Invalid relation indices: 0


## My observations

- Number of records loaded: 1,076
- Most common entity type: PhysicalObject (1,994)
- Most common relation type: hasParticipant/hasPatient (1,206)
- Interesting example: "air conditioner thermostat not working" shows how an equipment component and its condition are connected.
- Something I want to understand better: How can a model learn to identify entities and their relationships in maintenance records it has never seen before?

In [6]:

# Step 6: Inspect a few more maintenance records

for i, record in enumerate(records[1:5], start=2):
    print(f"\nRecord {i}: {record['text']}")

    for entity in record["entities"]:
        phrase = " ".join(
            record["tokens"][entity["start"]:entity["end"]]
        )
        category = entity["type"].split("/")[0]
        print(f"  {phrase} -> {category}")

    print("-" * 45)



Record 2: <id> air conditioner thermostat unserviceable
  air conditioner -> PhysicalObject
  thermostat -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Record 3: <id> air conditioner unserviceable
  air conditioner -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Record 4: <id> air conditioner unserviceable when stationary
  air conditioner -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Record 5: <id> air conditioner warm
  air conditioner -> PhysicalObject
  warm -> State
---------------------------------------------


In [7]:

# Step 7: Explore random maintenance records

import random

random.seed(42)

sample_records = random.sample(records, 5)

for i, record in enumerate(sample_records, start=1):
    print(f"\nExample {i}: {record['text']}")

    for entity in record["entities"]:
        phrase = " ".join(
            record["tokens"][entity["start"]:entity["end"]]
        )

        category = entity["type"].split("/")[0]

        print(f"  {phrase} -> {category}")

    print("-" * 45)



Example 1: <id> cracks in back right hand side handrail
  cracks -> Property
  handrail -> PhysicalObject
---------------------------------------------

Example 2: <id> beacon unserviceable
  beacon -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Example 3: <id> - change out jockey pulley bearing
  change out -> Activity
  jockey pulley -> PhysicalObject
  pulley -> PhysicalObject
  bearing -> PhysicalObject
---------------------------------------------

Example 4: <id> - CW right hand hoist cylinder leaking
  leaking -> Process
  hoist cylinder -> PhysicalObject
---------------------------------------------

Example 5: <id> replace timing sensor
  replace -> Activity
  timing sensor -> PhysicalObject
---------------------------------------------


In [8]:

# Step 8: Check how often entities overlap

overlap_records = []
overlap_pairs = 0

for record in records:
    entities = record["entities"]
    pairs = []

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            a = entities[i]
            b = entities[j]

            if max(a["start"], b["start"]) < min(a["end"], b["end"]):
                pairs.append((i, j))

    if pairs:
        overlap_records.append((record, pairs))
        overlap_pairs += len(pairs)

print("Total records:", len(records))
print("Records with overlapping entities:", len(overlap_records))
print("Overlapping entity pairs:", overlap_pairs)
print("Percentage of affected records:",
      round(100 * len(overlap_records) / len(records), 2), "%")

print("\nSome examples:")

for record, pairs in overlap_records[:3]:
    print("\nText:", record["text"])

    for i, j in pairs:
        a = record["entities"][i]
        b = record["entities"][j]

        phrase_a = " ".join(record["tokens"][a["start"]:a["end"]])
        phrase_b = " ".join(record["tokens"][b["start"]:b["end"]])

        print(f"  '{phrase_a}' overlaps with '{phrase_b}'")


Total records: 1076
Records with overlapping entities: 328
Overlapping entity pairs: 385
Percentage of affected records: 30.48 %

Some examples:

Text: <id> air filters plugged
  'air filters' overlaps with 'filters'

Text: <id> air horn disconnected
  'air horn' overlaps with 'horn'

Text: <id> air horn doesn't work
  'air horn' overlaps with 'horn'



## Findings from dataset exploration

- Loaded 1,076 maintenance records from MaintIE.
- Found five main entity categories.
- PhysicalObject is the most common entity category.
- Found 2,341 annotated relationships.
- All checked entity spans and relation indices were valid.
- Found 328 records containing overlapping entities (30.48%).
- Some records contain nested entities, such as "air filters" and "filters".

### Next steps

I plan to explore span-based entity recognition because overlapping entities cannot be fully represented using standard single-layer BIO tagging.

Before training, I also need to prepare the data splits and check for similar or repeated maintenance records that could affect evaluation.


In [9]:

# Step 9: Check for duplicate maintenance records

from collections import Counter

texts = [r["text"].strip().lower() for r in records]
counts = Counter(texts)

duplicates = {
    text: count
    for text, count in counts.items()
    if count > 1
}

print("Total records:", len(records))
print("Unique texts:", len(counts))
print("Repeated texts:", len(duplicates))
print("Extra duplicate occurrences:", len(records) - len(counts))

print("\nExamples:")
for text, count in list(duplicates.items())[:5]:
    print(f"{count} times: {text}")


Total records: 1076
Unique texts: 1074
Repeated texts: 2
Extra duplicate occurrences: 2

Examples:
2 times: <id> <num> v supply error
2 times: disconnect motor and reconnect new motor


In [10]:

# Step 10: Compare annotations in repeated records

from collections import defaultdict

groups = defaultdict(list)

for index, record in enumerate(records):
    text = record["text"].strip().lower()
    groups[text].append((index, record))

for text, entries in groups.items():
    if len(entries) > 1:
        print("\nRepeated text:", text)

        first_record = entries[0][1]

        for index, record in entries:
            print(f"\nRecord index: {index}")
            print("Entities:", record["entities"])
            print("Relations:", record["relations"])

        same_annotations = all(
            record["tokens"] == first_record["tokens"]
            and record["entities"] == first_record["entities"]
            and record["relations"] == first_record["relations"]
            for _, record in entries[1:]
        )

        print("\nExactly identical annotations:",
              same_annotations)
        print("-" * 50)



Repeated text: <id> <num> v supply error

Record index: 757
Entities: [{'start': 4, 'end': 5, 'type': 'State/UndesirableState'}, {'start': 1, 'end': 5, 'type': 'State/UndesirableState'}, {'start': 3, 'end': 5, 'type': 'State/UndesirableState'}]
Relations: [{'head': 1, 'tail': 2, 'type': 'isA'}, {'head': 2, 'tail': 0, 'type': 'isA'}]

Record index: 760
Entities: [{'start': 4, 'end': 5, 'type': 'State/UndesirableState'}, {'start': 1, 'end': 5, 'type': 'State/UndesirableState'}, {'start': 3, 'end': 5, 'type': 'State/UndesirableState'}]
Relations: [{'head': 1, 'tail': 2, 'type': 'isA'}, {'head': 2, 'tail': 0, 'type': 'isA'}]

Exactly identical annotations: False
--------------------------------------------------

Repeated text: disconnect motor and reconnect new motor

Record index: 874
Entities: [{'start': 0, 'end': 1, 'type': 'Activity/SupportingActivity/Isolate'}, {'start': 1, 'end': 2, 'type': 'PhysicalObject/DrivingObject/ElectromagneticRotationalDrivingObject'}, {'start': 3, 'end': 

In [11]:

# Step 11: Remove repeated texts from the working data

seen = set()
unique_records = []
removed_indices = []

for index, record in enumerate(records):
    text = record["text"].strip().lower()

    if text in seen:
        removed_indices.append(index)
    else:
        seen.add(text)
        unique_records.append(record)

print("Original records:", len(records))
print("Unique records:", len(unique_records))
print("Removed record indices:", removed_indices)


Original records: 1076
Unique records: 1074
Removed record indices: [760, 875]


In [12]:

# Step 12: Find similar maintenance records

def get_words(record):
    return {
        token.lower()
        for token in record["tokens"]
        if not (token.startswith("<") and token.endswith(">"))
    }

similar_pairs = []

for i in range(len(unique_records)):
    words_a = get_words(unique_records[i])

    if len(words_a) < 3:
        continue

    for j in range(i + 1, len(unique_records)):
        words_b = get_words(unique_records[j])

        if len(words_b) < 3:
            continue

        common = words_a & words_b
        total = words_a | words_b

        similarity = len(common) / len(total)

        if similarity >= 0.75 and len(common) >= 3:
            similar_pairs.append((similarity, i, j))

print("Potentially similar pairs:", len(similar_pairs))

print("\nSome examples:")

for score, i, j in sorted(
    similar_pairs, reverse=True
)[:10]:
    print(f"\nSimilarity: {score:.2f}")
    print("A:", unique_records[i]["text"])
    print("B:", unique_records[j]["text"])


Potentially similar pairs: 127

Some examples:

Similarity: 1.00
A: <id> change position <num> tyre
B: <id> change tyre position <num>

Similarity: 1.00
A: <id> change position <num> and <num> tyres
B: <id> change position <num> and position <num> tyres

Similarity: 1.00
A: <id> change out worn position <num> tyre
B: change out position <num> tyre worn

Similarity: 1.00
A: <id> change out position <num> <num> and <num> tyres
B: <id> change out position <num> and <num> tyres

Similarity: 1.00
A: <id> change out position <num> <num> <num> and <num> tyres
B: <id> change out position <num> and <num> tyres

Similarity: 1.00
A: <id> change out position <num> <num> <num> and <num> tyres
B: <id> change out position <num> <num> and <num> tyres

Similarity: 1.00
A: <id> brake cooling hose leaking
B: <id> leaking brake cooling hose

Similarity: 1.00
A: <id> blown position <num> tyre
B: position <num> tyre blown

Similarity: 0.89
A: <id> - change out universal joints engine - pump drive
B: <id> - 

In [13]:

# Step 13: Group potentially similar records

from collections import defaultdict

graph = defaultdict(set)

for score, i, j in similar_pairs:
    graph[i].add(j)
    graph[j].add(i)

visited = set()
groups = []

for start in graph:
    if start in visited:
        continue

    group = []
    stack = [start]

    while stack:
        node = stack.pop()

        if node in visited:
            continue

        visited.add(node)
        group.append(node)
        stack.extend(graph[node] - visited)

    groups.append(group)

groups.sort(key=len, reverse=True)

print("Records involved in similar pairs:", len(visited))
print("Groups of connected records:", len(groups))
print("Largest group size:", max(map(len, groups), default=0))

print("\nExamples from the largest group:")

if groups:
    for index in groups[0][:5]:
        print("-", unique_records[index]["text"])


Records involved in similar pairs: 156
Groups of connected records: 54
Largest group size: 11

Examples from the largest group:
- <id> change out blown position <num> tyre
- <id> change out position <num> tyre
- <id> change position <num> tyre
- <id> change position <num> flat tyre
- <id> change tyre position <num>


In [14]:

# Step 14: Group-aware train, validation and test split

from sklearn.model_selection import GroupShuffleSplit

# Give every record its own group initially
group_ids = list(range(len(unique_records)))

# Assign the same group ID to connected records
for group in groups:
    group_id = min(group)
    for index in group:
        group_ids[index] = group_id

indices = list(range(len(unique_records)))

# First split: approximately 80% train, 20% temporary
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)

train_idx, temp_idx = next(
    splitter.split(indices, groups=group_ids)
)

# Divide temporary records into validation and test
temp_groups = [group_ids[i] for i in temp_idx]

splitter2 = GroupShuffleSplit(
    n_splits=1, test_size=0.5, random_state=42
)

val_local, test_local = next(
    splitter2.split(temp_idx, groups=temp_groups)
)

val_idx = temp_idx[val_local]
test_idx = temp_idx[test_local]

train_records = [unique_records[i] for i in train_idx]
val_records = [unique_records[i] for i in val_idx]
test_records = [unique_records[i] for i in test_idx]

# Check group separation
split_groups = [
    {group_ids[i] for i in part}
    for part in (train_idx, val_idx, test_idx)
]

no_group_leakage = all(
    split_groups[i].isdisjoint(split_groups[j])
    for i in range(3)
    for j in range(i + 1, 3)
)

print("Training records:", len(train_records))
print("Validation records:", len(val_records))
print("Testing records:", len(test_records))
print("Total:", sum(map(len, [
    train_records, val_records, test_records
])))
print("No shared similarity groups:", no_group_leakage)


Training records: 868
Validation records: 107
Testing records: 99
Total: 1074
No shared similarity groups: True


In [15]:

# Step 15: Check entity distribution across dataset splits

from collections import Counter

def count_entities(data):
    counts = Counter()

    for record in data:
        for entity in record["entities"]:
            label = entity["type"].split("/")[0]
            counts[label] += 1

    return counts

train_counts = count_entities(train_records)
val_counts = count_entities(val_records)
test_counts = count_entities(test_records)

categories = sorted(
    set(train_counts) | set(val_counts) | set(test_counts)
)

print(f"{'Category':<18} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 46)

for category in categories:
    print(
        f"{category:<18}"
        f"{train_counts[category]:>8}"
        f"{val_counts[category]:>8}"
        f"{test_counts[category]:>8}"
    )

print("\nMissing categories:")

missing = False

for category in categories:
    if (train_counts[category] == 0
        or val_counts[category] == 0
        or test_counts[category] == 0):
        print(category)
        missing = True

if not missing:
    print("None")


Category              Train      Val     Test
----------------------------------------------
Activity               628      84      70
PhysicalObject        1614     199     179
Process                122       9      15
Property                32       3       0
State                  338      43      54

Missing categories:
Property


In [16]:

# Step 16: Examine Property records and their groups

from collections import Counter

property_indices = []

for i, record in enumerate(unique_records):
    has_property = any(
        entity["type"].split("/")[0] == "Property"
        for entity in record["entities"]
    )

    if has_property:
        property_indices.append(i)

property_groups = Counter(
    group_ids[i] for i in property_indices
)

print("Records containing Property:", len(property_indices))
print("Distinct groups containing Property:", len(property_groups))

print("\nProperty records in each split:")

for name, indices_part in [
    ("Training", train_idx),
    ("Validation", val_idx),
    ("Testing", test_idx)
]:
    count = sum(
        i in property_indices
        for i in indices_part
    )
    print(f"{name}: {count}")

print("\nSome Property examples:")

for i in property_indices[:5]:
    print("-", unique_records[i]["text"])


Records containing Property: 35
Distinct groups containing Property: 35

Property records in each split:
Training: 32
Validation: 3
Testing: 0

Some Property examples:
- <id> crack in blade liner left hand side
- <id> crack in boom <id> chord
- <id> crack in fuel tank
- <id> crack in headboard
- <id> crack in lower centre of windscreen


In [17]:

# Step 17: Create a group-aware split with rare-class coverage

import random
from collections import defaultdict, Counter

# Collect records belonging to each similarity group
group_members = defaultdict(list)

for i, group_id in enumerate(group_ids):
    group_members[group_id].append(i)

# Find groups containing Property annotations
property_groups = []

for group_id, indices in group_members.items():
    if any(
        entity["type"].split("/")[0] == "Property"
        for i in indices
        for entity in unique_records[i]["entities"]
    ):
        property_groups.append(group_id)

other_groups = [
    g for g in group_members if g not in set(property_groups)
]

# Shuffle reproducibly
rng = random.Random(42)
rng.shuffle(property_groups)
rng.shuffle(other_groups)

new_split = {"train": [], "val": [], "test": []}

def add_group(group_id, split_name):
    new_split[split_name].extend(group_members[group_id])

# Reserve Property-containing groups for each split
for g in property_groups[:4]:
    add_group(g, "test")

for g in property_groups[4:8]:
    add_group(g, "val")

for g in property_groups[8:]:
    add_group(g, "train")

# Fill the remaining groups toward an 80/10/10 split
n = len(unique_records)

targets = {
    "train": round(n * 0.8),
    "val": round(n * 0.1),
    "test": n - round(n * 0.8) - round(n * 0.1)
}

for g in other_groups:
    destination = max(
        targets,
        key=lambda s: targets[s] - len(new_split[s])
    )
    add_group(g, destination)

# Validate the new split
all_indices = sum(new_split.values(), [])

assert len(all_indices) == n
assert len(set(all_indices)) == n

split_group_sets = {
    s: {group_ids[i] for i in indices}
    for s, indices in new_split.items()
}

assert all(
    split_group_sets[a].isdisjoint(split_group_sets[b])
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]
)

print("New dataset split:")
for name, indices in new_split.items():
    data = [unique_records[i] for i in indices]
    counts = Counter(
        e["type"].split("/")[0]
        for r in data
        for e in r["entities"]
    )

    print(
        name,
        "| Records:", len(indices),
        "| Property:", counts["Property"],
        "| Categories:", len(counts)
    )

print("Total records:", len(all_indices))
print("Group separation: Passed")


New dataset split:
train | Records: 859 | Property: 27 | Categories: 5
val | Records: 107 | Property: 4 | Categories: 5
test | Records: 108 | Property: 4 | Categories: 5
Total records: 1074
Group separation: Passed


In [18]:

# Step 18: Check the final entity distribution

from collections import Counter

final_records = {
    name: [unique_records[i] for i in indices]
    for name, indices in new_split.items()
}

def count_categories(data):
    counts = Counter()

    for record in data:
        for entity in record["entities"]:
            category = entity["type"].split("/")[0]
            counts[category] += 1

    return counts

train_counts = count_categories(final_records["train"])
val_counts = count_categories(final_records["val"])
test_counts = count_categories(final_records["test"])

categories = sorted(
    set(train_counts) | set(val_counts) | set(test_counts)
)

print(f"{'Category':<18} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 46)

for category in categories:
    print(
        f"{category:<18}"
        f"{train_counts[category]:>8}"
        f"{val_counts[category]:>8}"
        f"{test_counts[category]:>8}"
    )

print("\nAll categories represented:",
      all(
          train_counts[c] > 0
          and val_counts[c] > 0
          and test_counts[c] > 0
          for c in categories
      ))


Category              Train      Val     Test
----------------------------------------------
Activity               630      84      68
PhysicalObject        1605     200     187
Process                114      16      16
Property                27       4       4
State                  340      37      58

All categories represented: True


In [19]:

# Step 19: Check overlapping entities in each split

def count_overlaps(data):
    affected_records = 0
    overlap_pairs = 0

    for record in data:
        entities = record["entities"]
        record_has_overlap = False

        for i in range(len(entities)):
            for j in range(i + 1, len(entities)):
                a = entities[i]
                b = entities[j]

                if max(a["start"], b["start"]) < min(a["end"], b["end"]):
                    overlap_pairs += 1
                    record_has_overlap = True

        if record_has_overlap:
            affected_records += 1

    return affected_records, overlap_pairs


for name in ["train", "val", "test"]:
    data = final_records[name]

    affected, pairs = count_overlaps(data)

    percentage = 100 * affected / len(data)

    print(f"\n{name.upper()}")
    print("Total records:", len(data))
    print("Records with overlaps:", affected)
    print("Overlapping pairs:", pairs)
    print(f"Percentage: {percentage:.2f}%")



TRAIN
Total records: 859
Records with overlaps: 256
Overlapping pairs: 294
Percentage: 29.80%

VAL
Total records: 107
Records with overlaps: 33
Overlapping pairs: 47
Percentage: 30.84%

TEST
Total records: 108
Records with overlaps: 38
Overlapping pairs: 41
Percentage: 35.19%


In [20]:

# Step 20: Save reproducible dataset splits

import json
import hashlib
from google.colab import files

# Create a fingerprint of our deduplicated dataset
dataset_hash = hashlib.sha256(
    json.dumps(
        unique_records,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

# Save the exact record indices for each split
split_data = {
    "dataset": "MaintIE gold_release",
    "source": "https://github.com/nlp-tlp/maintie",
    "total_records": len(unique_records),
    "dataset_sha256": dataset_hash,
    "random_seed": 42,
    "method": "Group-aware split with Property coverage",
    "splits": {
        name: sorted(int(i) for i in indices)
        for name, indices in new_split.items()
    }
}

filename = "forgegraph_split_v1.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(split_data, f, indent=2)

print("Saved:", filename)

for name, indices in split_data["splits"].items():
    print(f"{name}: {len(indices)} records")

print("Dataset fingerprint:", dataset_hash[:16])

files.download(filename)


Saved: forgegraph_split_v1.json
train: 859 records
val: 107 records
test: 108 records
Dataset fingerprint: 9912429ac4d4d9ab


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>